## Notebook Summary

This notebook sets up a registry-driven pattern for Cosmos ingestion sources.

- Creates `cosmos_poc.ops.ingestion_sources` to store source configuration and operational metadata.
- Seeds the registry with example Cosmos DB sources.
- Uses the registry as the control plane for which sources are ingested and how they are authenticated.

### Design Notes

- **Per-source secret names:** Each source can point to its own service principal credentials, including sources in different accounts or tenants. No code changes are required; only the registry row differs.
- **Onboarding and pausing:** To onboard a new source, insert a row into the registry. To pause a source, set `enabled = false`.
- **Identifiers vs. secrets:** `tenant_id` and `subscription_id` are stored in the registry because they are identifiers, not secrets. Credentials remain in Key Vault via the configured secret scope and secret names.
- **Access model:** The service principal already has read access to the registry (`SELECT` on `ops` granted earlier), while write access is restricted to admins.


In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS cosmos_poc.ops.ingestion_sources (
  source_id            STRING  NOT NULL COMMENT 'Stable unique id, e.g. <database>_<container>',
  enabled              BOOLEAN NOT NULL COMMENT 'Only enabled sources are processed',
  cosmos_endpoint      STRING  NOT NULL,
  tenant_id            STRING  NOT NULL,
  subscription_id      STRING  NOT NULL,
  resource_group       STRING  NOT NULL,
  cosmos_database      STRING  NOT NULL,
  cosmos_container     STRING  NOT NULL,
  secret_scope         STRING  NOT NULL COMMENT 'Databricks secret scope holding the SP credentials for this source',
  client_id_secret     STRING  NOT NULL COMMENT 'Secret name for the SP client id',
  client_secret_secret STRING  NOT NULL COMMENT 'Secret name for the SP client secret',
  start_from           STRING           COMMENT 'First run only: Beginning | Now | ISO-8601',
  max_items_per_batch  INT,
  owner                STRING,
  notes                STRING,
  updated_at           TIMESTAMP,
  CONSTRAINT ingestion_sources_pk PRIMARY KEY (source_id)
)
COMMENT 'Registry of Cosmos sources for the raw/bronze ingestion framework'
""")

In [0]:
TENANT_ID       = "f4684704-cde2-4f4d-8eb0-be045ed13a2c"
SUBSCRIPTION_ID = "d93f4f31-c264-4bb9-aa6a-efa162124a9e"
EP = "https://cosmos-dbx-poc-sm.documents.azure.com:443/"

cols = ("source_id STRING, enabled BOOLEAN, cosmos_endpoint STRING, tenant_id STRING, subscription_id STRING, "
        "resource_group STRING, cosmos_database STRING, cosmos_container STRING, secret_scope STRING, "
        "client_id_secret STRING, client_secret_secret STRING, start_from STRING, max_items_per_batch INT, "
        "owner STRING, notes STRING")
rows = [
    ("pocdb_customers", True, EP, TENANT_ID, SUBSCRIPTION_ID, "rg-cosmos-dbx-poc", "pocdb", "customers",
     "kv-poc", "sp-cosmos-client-id", "sp-cosmos-client-secret", "Beginning", 100000, "data-eng", "Customer master"),
    ("pocdb_orders",    True, EP, TENANT_ID, SUBSCRIPTION_ID, "rg-cosmos-dbx-poc", "pocdb", "orders",
     "kv-poc", "sp-cosmos-client-id", "sp-cosmos-client-secret", "Beginning", 100000, "data-eng", "Orders - nested lines"),
]
(spark.createDataFrame(rows, cols)
     .selectExpr("*", "current_timestamp() AS updated_at")
     .createOrReplaceTempView("_seed"))

spark.sql("""
MERGE INTO cosmos_poc.ops.ingestion_sources t USING _seed s ON t.source_id = s.source_id
WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *""")

display(spark.sql("""SELECT source_id, enabled, cosmos_database, cosmos_container, secret_scope, start_from
                     FROM cosmos_poc.ops.ingestion_sources ORDER BY source_id"""))

In [0]:
display(spark.sql("""SELECT source_id, right(tenant_id, 4) AS tenant_last4, right(subscription_id, 4) AS sub_last4
                     FROM cosmos_poc.ops.ingestion_sources"""))